This script compares Deberta to Roberta for Task 1

In [3]:
!pip install -q transformers datasets scikit-learn accelerate torch pandas

import random
import numpy as np
import torch
import torch.nn as nn
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)
from datasets import load_dataset
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

# Configuration
SEED = 42
MODELS = ["microsoft/deberta-v3-large", "roberta-large"]
OUTPUT_DIR = "./model_comparison"
MAX_LEN = 512
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
LR = 8e-6
EPOCHS = 15

def set_deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_deterministic(SEED)
set_seed(SEED)

# Data Preparation
dataset = load_dataset("ailsntua/QEvasion")

def preprocess(example):
    text = f"Question: {example['question']} Answer: {example['interview_answer']}"
    return {"text": text, "clarity_label": example["clarity_label"]}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": (preds == labels).mean(),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

full_data = dataset["train"].map(preprocess)
full_data = full_data.class_encode_column("clarity_label")

# Split Data
split1 = full_data.train_test_split(
    test_size=0.1, seed=SEED, stratify_by_column="clarity_label"
)
train_dev_ds = split1["train"]
held_out_test_ds = split1["test"]

split2 = train_dev_ds.train_test_split(
    test_size=0.1, seed=SEED, stratify_by_column="clarity_label"
)
train_ds = split2["train"]
eval_ds = split2["test"]

labels = train_ds.features["clarity_label"].names
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for name, i in label2id.items()}

# Class Weights
y_train = train_ds["clarity_label"]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights_tensor,
            label_smoothing=0.1,
        )
        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss

# Training Loop
final_metrics = []

for model_name in MODELS:
    set_deterministic(SEED)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
        )

    current_train_ds = train_ds.map(tokenize_fn, batched=True)
    current_eval_ds = eval_ds.map(tokenize_fn, batched=True)
    current_test_ds = held_out_test_ds.map(tokenize_fn, batched=True)

    for ds in [current_train_ds, current_eval_ds, current_test_ds]:
        ds.set_format(type="torch", columns=["input_ids", "attention_mask", "clarity_label"])

    current_train_ds = current_train_ds.map(lambda x: {"labels": x["clarity_label"]})
    current_eval_ds = current_eval_ds.map(lambda x: {"labels": x["clarity_label"]})
    current_test_ds = current_test_ds.map(lambda x: {"labels": x["clarity_label"]})

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
    )

    safe_name = model_name.replace("/", "-")

    training_args = TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/{safe_name}",
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACCUMULATION,
        num_train_epochs=EPOCHS,
        weight_decay=0.05,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=True,
        report_to="none",
        seed=SEED,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=current_train_ds,
        eval_dataset=current_eval_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
    )

    trainer.train()

    print(f"Eval on {model_name}")
    test_results = trainer.evaluate(current_test_ds)

    final_metrics.append({
        "Model": model_name,
        "Macro_F1": test_results["eval_macro_f1"],
        "Accuracy": test_results["eval_accuracy"],
        "Loss": test_results["eval_loss"]
    })

    del model
    del trainer
    torch.cuda.empty_cache()

# Results
df_results = pd.DataFrame(final_metrics)
df_results = df_results.sort_values(by="Macro_F1", ascending=False).reset_index(drop=True)

print(df_results)

best_model = df_results.iloc[0]
print(f"\nBest Performing Model: {best_model['Model']}")
print(f"Best Macro F1: {best_model['Macro_F1']:.4f}")

Map:   0%|          | 0/3448 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/3448 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/2792 [00:00<?, ? examples/s]

Map:   0%|          | 0/311 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/345 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.170721,0.398714,0.358257
2,No log,1.015699,0.630225,0.610613
3,1.123700,0.995983,0.675241,0.621242
4,1.123700,0.939414,0.684887,0.651578
5,1.123700,0.983783,0.681672,0.661916
6,0.806700,1.048648,0.633441,0.620642
7,0.806700,1.070758,0.684887,0.660367
8,0.806700,1.154491,0.646302,0.636563
9,0.610000,1.155900,0.684887,0.650005


Eval on roberta-large


                        Model  Macro_F1  Accuracy      Loss
0               roberta-large  0.684646  0.698551  0.988386
1  microsoft/deberta-v3-large  0.682496  0.684058  0.979747

Best Performing Model: roberta-large
Best Macro F1: 0.6846


In [2]:
!pip install -q transformers datasets scikit-learn accelerate torch pandas

import random
import numpy as np
import torch
import torch.nn as nn
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)
from datasets import load_dataset
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

# seeding
SEED = 42

def set_deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_deterministic(SEED)
set_seed(SEED)

# configuration
MODELS = ["microsoft/deberta-v3-large", "roberta-large"]
OUTPUT_DIR = "./model_comparison"
MAX_LEN = 512
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
LR = 8e-6
EPOCHS = 15

# data
dataset = load_dataset("ailsntua/QEvasion")

def preprocess(example):
    # Task 1: predicting clarity_label
    # removed label from input text to prevent leakage
    text = f"Question: {example['question']} Answer: {example['interview_answer']}"
    return {"text": text, "clarity_label": example["clarity_label"]}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": (preds == labels).mean(),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

full_data = dataset["train"].map(preprocess)
full_data = full_data.class_encode_column("clarity_label")

# train / dev / held-out split
# splitting once ensures both models see exact same data
split1 = full_data.train_test_split(
    test_size=0.1,
    seed=SEED,
    stratify_by_column="clarity_label",
)
train_dev_ds = split1["train"]
held_out_test_ds = split1["test"]

split2 = train_dev_ds.train_test_split(
    test_size=0.1,
    seed=SEED,
    stratify_by_column="clarity_label",
)
train_ds = split2["train"]
eval_ds = split2["test"]

labels = train_ds.features["clarity_label"].names
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for name, i in label2id.items()}

# class weights
y_train = train_ds["clarity_label"]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)


# custom trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights_tensor,
            label_smoothing=0.1,
        )
        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss


# loop through models
results_log = {}

for model_name in MODELS:
    print(f"Training the {model_name} model")
    set_deterministic(SEED)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
        )

    # re-tokenize for current model
    current_train_ds = train_ds.map(tokenize_fn, batched=True)
    current_eval_ds = eval_ds.map(tokenize_fn, batched=True)
    current_test_ds = held_out_test_ds.map(tokenize_fn, batched=True)

    current_train_ds = current_train_ds.map(lambda x: {"labels": x["clarity_label"]})
    current_eval_ds = current_eval_ds.map(lambda x: {"labels": x["clarity_label"]})
    current_test_ds = current_test_ds.map(lambda x: {"labels": x["clarity_label"]})

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
    )

    training_args = TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/{model_name}",
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACCUMULATION,
        num_train_epochs=EPOCHS,
        weight_decay=0.05,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=True,
        report_to="none",
        seed=SEED,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=current_train_ds,
        eval_dataset=current_eval_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
    )

    trainer.train()

    # evaluation
    test_results = trainer.evaluate(current_test_ds)
    results_log[model_name] = test_results

    print(f"{model_name} final macro-F1:", round(test_results["eval_macro_f1"], 4))
    print(f"{model_name} final accuracy:", round(test_results["eval_accuracy"], 4))

Map:   0%|          | 0/3448 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/3448 [00:00<?, ? examples/s]

Training the microsoft/deberta-v3-large model


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/2792 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/345 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.193013,0.379421,0.349019
2,No log,1.033942,0.607717,0.601575
3,1.120500,0.965090,0.681672,0.647990
4,1.120500,0.954029,0.684887,0.651554
5,1.120500,1.007150,0.684887,0.650026
6,0.803100,1.057873,0.684887,0.663610
7,0.803100,1.066535,0.697749,0.677990
8,0.803100,1.045328,0.700965,0.676196
9,0.616900,1.139045,0.710611,0.668290
10,0.616900,1.152355,0.710611,0.675382


roberta-large final macro-F1: 0.6614
roberta-large final accuracy: 0.6667
